# Module 03: EDA

In [ ]:
# packages
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
from sklearn.model_selection import train_test_split 
from ISLP import load_data

# set seed
seed = 2323

### We'll use the _Hitters_ data from ISLP for this activity. The metadata for _Hitters_ can be found [here](https://intro-stat-learning.github.io/ISLP/datasets/Hitters.html).

In [ ]:
# Load the data
Hitters = load_data('Hitters')

### Determine the number of rows and columns in the dataset by returning its "shape" attribute

In [ ]:
Hitters.shape

### Determine whether each feature is numeric or categorical by returning the "dtype" attribute for each column

In [ ]:
for col in Hitters.columns:
    print(col, Hitters[col].dtype)

### Before doing any other analyses, let's create training and test sets.

In [ ]:
Train, Test = train_test_split(Hitters, 
                               random_state=seed, 
                               test_size=0.40, 
                               shuffle=True) 

### Based on the metadata, what is the difference between the 6 columns starting with 'C' and the 6 related columns that don't?

The six columns starting with 'C'  show the player's career totals through 1986, while the corresponding six columns without the 'C' prefix show only the totals for the 1986 season.

### On the training set, create pairwise scatterplots for each of these 6 columns with the 'Salary' variable.

In [ ]:
# First create a subset of the columns that we want to plot

subset = Train[['CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks']]

# Initialize the plots before drawing them

fig, axes = subplots(nrows=2,
                     ncols=3,
                     figsize=(15, 10))
# Copy the helper function

def range_to_grid(i,Ncol):
    x=[]
    y=[]
    for n in range(Ncol**2):
        x.append(int(np.floor(n/Ncol)))
        y.append(n % Ncol)
        #print(n,x[n],y[n])
    return x[i],y[i]

# Plot the variables

for j in range(len(subset.columns)):
    axes[range_to_grid(j,3)[0],range_to_grid(j,3)[1]].plot(subset.iloc[:,j], Train['Salary'], 'o')
    axes[range_to_grid(j,3)[0],range_to_grid(j,3)[1]].set_xlabel(subset.columns[j])


### Use the "describe" method to determine the mean, standard deviation, and 5 number summary of all numeric variables in the training subset of _Hitters_.

In [ ]:
Train.describe()

### It looks like the mean and median of 'AtBat' are nearly equal. This _might_ suggest that this variable is normally distributed. Create a histogram of 'AtBat' to check this hypothesis.

In [ ]:
plt.hist(Train['AtBat'], bins=20)
plt.xlabel('AtBat')
plt.ylabel('Frequency')
plt.title('Histogram of AtBat (Training Set)')
plt.show()

### Let's standardize the AtBat feature (i.e., normalize by z-scores). We'll create a new column in the training data called 'AtBat_st' to represent this.

In [ ]:
Train['AtBat_st'] = (Train['AtBat'] - Train['AtBat'].mean()) / Train['AtBat'].std()

### How many rows have an 'AtBat' value within the first standard deviation?

Hint: the 'len' magic method returns the number of rows of a dataFrame.

In [ ]:
len(Train[(Train['AtBat_st'] >= -1) & (Train['AtBat_st'] <= 1)])

### Going back to the results of the 'describe' method, how can you tell that the 'Salary' variable has missing values?

In the 'describe' output, the 'count' for Salary is smaller than the count for the other numeric columns. Since 'describe' only counts non-missing values, a lower count for Salary shows  some values are missing.

### Describe a situation where a variable could have missing values but this would not be reflected in the results of the 'describe' method.

If missing values were encoded using a sentinel value instead of true NaN, pandas would treat those entries as valid observations. 'describe' would then include them in the count and wouldn't flag anything as missing even though the values are not real.

### On the training data, create separate boxplots of the 'AtBat' variable for when 'Salary' is populated or missing.

In [ ]:
Train['Salary_missing'] = Train['Salary'].isna().map({True: 'Missing', False: 'Populated'})

fig, ax = subplots(figsize=(6, 5))
Train.boxplot(column='AtBat', by='Salary_missing', ax=ax)
ax.set_xlabel('Salary status')
ax.set_ylabel('AtBat')
plt.title('AtBat by Salary Missingness')
plt.suptitle('')
plt.show()

### Create a correlation matrix for all numeric features in the training set

In [ ]:
Train.corr(numeric_only=True)

### Propose two different ways of imputing the missing values of Salary while taking advantage of the information given in the boxplots or the correlation matrix.

One way would be Regression imputation where the Salary is fairly strongly correlated with the career statistics. We could fit a linear regression of Salary on a variable like CAtBat or CRBI using players with non-missing Salary, then use that model to predict Salary for players with missing values.
Another way is Group-based or binnedimputation: The boxplots show AtBat differs  between players with missing vs. populated Salary. We could bin players by AtBat (or a similar variable) and impute each missing Salary with the median Salary of players in the same bin, instead of using a single overall median for everyone.

### For our last exercise, we'll explore Hits and Walks relative to AtBat totals. 
- Use the sum function to calculuate the totals of each of these three variables for the 1986 season (on the training set). 
- Create a pie chart which shows total hits, total walks, and remaining total (neither) as percents of the At Bats total (on the training set). 

In [ ]:
TotHits = Train['Hits'].sum()
TotWalks = Train['Walks'].sum()
TotAtBat = Train['AtBat'].sum()

Labels = ['Hits', 'Walks', 'Neither']
Totals = [TotHits, TotWalks, TotAtBat-TotHits-TotWalks]

In [ ]:
# pie chart
plt.pie(Totals, labels=Labels, autopct='%1.1f%%')
plt.title('1986 AtBat Composition (Training Set)')
plt.show()


### The previous two cells gave us totals across all players. For each player in the training set, calculate the Hits as a percent of AtBat and store it in a new variable called 'AVG'

In [ ]:
Train['AVG'] = Train['Hits'] / Train['AtBat']

### Using 0.25 and 0.31 as the split points, create a new variable with three bins: high, medium, and low. 

In [ ]:
Train['AVG_bin'] = 'medium'
Train.loc[Train['AVG'] < 0.25, 'AVG_bin'] = 'low'
Train.loc[Train['AVG'] > 0.31, 'AVG_bin'] = 'high'

### Create a bar chart that displays the number of players in each of the low, medium, and high categories (for the training data).

In [ ]:
Train['AVG_bin'].value_counts()

Notice that the order of the bars will be medium, low, high. That's counterintuitive. We can reorder these quickly. 

In [ ]:
indexMap = ['low', 'medium', 'high']
reordered_list = [Train['AVG_bin'].value_counts()[i] for i in indexMap]

In [ ]:
plt.bar(indexMap, reordered_list)

plt.title("1986 AVG (Training Set)")
plt.ylabel("Number of Players")

plt.xticks(range(len(indexMap)), indexMap)

plt.show()

### Did we use the depth method or width method for creating these bins? Explain.

This is the width method (equal-width binning). The bin boundaries (0.25 and 0.31) were fixed values on the AVG scale instead of values chosen to make each bin contain an equal number of players. The depth method (equal-frequency/quantile binning) would  choose  points so that each bin held  the same number of players, and those cut points would depend on the data's distribution rather than being fixed in advance.